# 14 — The GroupBy Object: The Split-Apply-Combine Engine
> **Interview Prep & Technical Mastery Guide**
> 
> *A comprehensive, battle-tested reference for Python Data Science, SQL-to-Pandas Translation, and Data Engineering Interviews.*

---

## 📌 Executive Summary & Interview Expectations
The **Split-Apply-Combine** paradigm is the most heavily tested aggregation concept across all data interviews. Interviewers test whether you understand:
1. **Lazy Execution**: Why `df.groupby()` is an iterator/generator that executes no computations until an aggregation method is called.
2. **`size()` vs `count()`**: The exact SQL equivalent of `COUNT(*)` vs `COUNT(column)` and their null-handling mechanics.
3. **Targeted Aggregation (`agg`)**: Named aggregations, dictionary-based multi-column mappings, and multi-metric summaries.
4. **The `as_index=False` Parameter**: Writing cleaner pipelines without redundant `.reset_index()` calls.
5. **`.transform()` vs `.agg()`**: Broadcasting group metrics back to the original DataFrame's index (critical for z-scores, normalization, and percent-of-total metrics).
6. **Interview Corner**: Group filtering with `.filter()`, windowing with `.transform()`, and market concentration drills.

## 1. Environment Setup & GroupBy Mechanics from Scratch

In [1]:
import os
import numpy as np
import pandas as pd

# Creating a toy supermarket inventory
food_data = {
    "Item": ["Banana", "Cucumber", "Orange", "Tomato", "Watermelon"],
    "Type": ["Fruit", "Vegetable", "Fruit", "Vegetable", "Fruit"],
    "Price": [0.99, 1.25, 0.49, 1.75, 3.45]
}

supermarket = pd.DataFrame(food_data)
supermarket

,Item,Type,Price
0,Banana,Fruit,0.99
1,Cucumber,Vegetable,1.25
2,Orange,Fruit,0.49
3,Tomato,Vegetable,1.75
4,Watermelon,Fruit,3.45


In [2]:
# 1. Split: Create lazy GroupBy object
type_groups = supermarket.groupby("Type")
print("GroupBy Object Type:", type(type_groups))

# Inspect individual groups
print("\nGroup 'Fruit':")
display(type_groups.get_group("Fruit"))

# 2. Apply & Combine: Mean price per food type
display(type_groups["Price"].mean())

GroupBy Object Type: <class 'pandas.api.typing.DataFrameGroupBy'>

Group 'Fruit':


,Item,Type,Price
0,Banana,Fruit,0.99
2,Orange,Fruit,0.49
4,Watermelon,Fruit,3.45


Type
Fruit        1.643333
Vegetable    1.500000
Name: Price, dtype: float64

## 2. Ingesting Real-World Corporate Data: `fortune1000.csv`

In [3]:
# Load Fortune 1000 dataset
csv_path = "fortune1000.csv"
if not os.path.exists(csv_path):
    csv_path = "https://raw.githubusercontent.com/paskhaver/pandas-in-action/master/chapter_09_the_groupby_object/fortune1000.csv"

fortune = pd.read_csv(csv_path)
print("Fortune 1000 loaded. Shape:", fortune.shape)
fortune.head(3)

Fortune 1000 loaded. Shape: (1000, 6)


,Company,Revenues,Profits,Employees,Sector,Industry
0,Walmart,500343.0,9862.0,2300000,Retailing,General Merchandisers
1,Exxon Mobil,244363.0,19710.0,71200,Energy,Petroleum Refining
2,Berkshire Hathaway,242137.0,44940.0,377000,Financials,Insurance: Property and Casualty (Stock)


## 3. Structural Attributes & Inspection of GroupBy Objects

### ⚠️ Top Interview Question: `sectors.size()` vs `sectors.count()`
- `sectors.size()`: Returns total **rows** per group, **including nulls** ($O(1)$ group metadata). Equivalent to SQL `COUNT(*)`.
- `sectors.count()`: Returns number of **non-null values** per column within each group. Equivalent to SQL `COUNT(col)`.
- `len(sectors)`: Returns total number of distinct groups (identical to `df['Sector'].nunique()`).

In [4]:
# Group by Sector
sectors = fortune.groupby("Sector")

print(f"Number of distinct sectors: {len(sectors)} (matches nunique: {fortune['Sector'].nunique()})")
print("\nTotal companies per sector (sectors.size()):")
display(sectors.size().sort_values(ascending=False).head(5))

Number of distinct sectors: 21 (matches nunique: 21)

Total companies per sector (sectors.size()):


Sector
Financials     155
Energy         107
Technology     103
Retailing       77
Health Care     71
dtype: int64

In [5]:
# Inspect underlying dictionary mapping groups to index row labels
sample_groups_dict = {k: sectors.groups[k][:3] for k in list(sectors.groups.keys())[:3]}
print("Sample groups mapping (Sector -> Row Indices):")
print(sample_groups_dict)

Sample groups mapping (Sector -> Row Indices):
{'Aerospace & Defense': Index([26, 50, 58], dtype='int64'), 'Apparel': Index([88, 241, 331], dtype='int64'), 'Business Services': Index([142, 160, 187], dtype='int64')}


## 4. Basic Aggregations: Sum, Mean, and Slicing SeriesGroupBy

In [6]:
# Top 5 sectors by total revenue
sectors["Revenues"].sum().sort_values(ascending=False).head(5)

Sector
Financials     2442480.0
Retailing      1684353.0
Energy         1543507.2
Health Care    1507991.4
Technology     1374822.3
Name: Revenues, dtype: float64

In [7]:
# Average employee count per company by sector
sectors["Employees"].mean().round(1).sort_values(ascending=False).head(5)

Sector
Food &  Drug Stores              116506.2
Hotels, Restaurants & Leisure     88628.3
Retailing                         88320.2
Telecommunications                78560.9
Motor Vehicles & Parts            49662.2
Name: Employees, dtype: float64

## 5. Multi-Column Grouping: Sector & Industry Breakdown

In [8]:
# Group by Sector AND Industry
sector_industry = fortune.groupby(["Sector", "Industry"])
print(f"Total Sector-Industry sub-segments: {len(sector_industry)}")

# Sum revenues per sub-segment
subsegment_revenues = sector_industry["Revenues"].sum().sort_values(ascending=False)
subsegment_revenues.head(6)

Total Sector-Industry sub-segments: 82


Sector       Industry                                
Retailing    General Merchandisers                       801826.0
Energy       Petroleum Refining                          710508.0
Financials   Commercial Banks                            679885.0
             Insurance: Property and Casualty (Stock)    595016.0
Health Care  Health Care: Insurance and Managed Care     541333.0
Wholesalers  Wholesalers: Health Care                    509026.0
Name: Revenues, dtype: float64

## 6. Advanced Aggregations: Named Aggregation & Dictionaries

### 💡 Modern Pandas Standard: Named Aggregation
Instead of older un-named multi-indexes, use tuple syntax:
```python
df.groupby('Sector').agg(
    total_rev=('Revenues', 'sum'),
    avg_profit=('Profits', 'mean'),
    firm_count=('Company', 'count')
)
```
This generates clean, single-level column headers automatically!

In [9]:
# Named aggregation syntax
sector_summary = (
    fortune.groupby("Sector")
    .agg(
        company_count=("Company", "count"),
        total_revenue=("Revenues", "sum"),
        avg_profit=("Profits", "mean"),
        median_employees=("Employees", "median")
    )
    .sort_values(by="total_revenue", ascending=False)
)

sector_summary.head(5).round(2)

,company_count,total_revenue,avg_profit,median_employees
Sector,,,,
Financials,155,2442480.0,1704.86,7570.0
Retailing,77,1684353.0,723.47,24200.0
Energy,107,1543507.2,805.37,4565.0
Health Care,71,1507991.4,1306.92,26600.0
Technology,103,1374822.3,1743.96,13000.0


## 7. ⚠️ Top Interview Concept: `.transform()` vs `.agg()`

| Property | `.agg()` | `.transform()` |
| :--- | :--- | :--- |
| **Output Shape** | Reduced (1 row per group) | **Preserved (Same row count as original DataFrame)** |
| **Return Type** | Aggregated DataFrame / Series | Broadcasted Series / DataFrame |
| **Primary Use Cases** | Summary tables, business reporting | Feature engineering, normalization, z-scores, market share |

In [10]:
# Using .transform() to calculate percentage contribution of each firm to its sector
fortune_enhanced = fortune.copy()

# 1. Calculate sector total revenue broadcasted to every row
fortune_enhanced["Sector_Total_Rev"] = fortune.groupby("Sector")["Revenues"].transform("sum")

# 2. Compute firm's percentage share of its sector
fortune_enhanced["Market_Share_Pct"] = (
    (fortune_enhanced["Revenues"] / fortune_enhanced["Sector_Total_Rev"]) * 100
).round(2)

fortune_enhanced[["Company", "Sector", "Revenues", "Sector_Total_Rev", "Market_Share_Pct"]].head(6)

,Company,Sector,Revenues,Sector_Total_Rev,Market_Share_Pct
0,Walmart,Retailing,500343.0,1684353.0,29.71
1,Exxon Mobil,Energy,244363.0,1543507.2,15.83
2,Berkshire Hathaway,Financials,242137.0,2442480.0,9.91
3,Apple,Technology,229234.0,1374822.3,16.67
4,UnitedHealth Group,Health Care,201159.0,1507991.4,13.34
5,McKesson,Wholesalers,198533.0,888149.0,22.35


## 8. GroupBy Cheat Sheet

| Task | Idiomatic Syntax | Key Benefit |
| :--- | :--- | :--- |
| **Count rows (incl nulls)** | `df.groupby('A').size()` | Fast $O(1)$ group metadata; SQL `COUNT(*)` |
| **Count valid values** | `df.groupby('A')['B'].count()` | Ignores nulls; SQL `COUNT(B)` |
| **Keep as Columns** | `df.groupby('A', as_index=False).sum()` | Eliminates redundant `.reset_index()` |
| **Named Aggregation** | `df.groupby('A').agg(total=('B', 'sum'))` | Creates clean single-tier column names |
| **Broadcast to Rows** | `df.groupby('A')['B'].transform('mean')` | Retains original index and shape |
| **Group Filtering** | `df.groupby('A').filter(lambda g: len(g) > 10)` | Filters entire groups by aggregate logic |

---
## 🎯 9. Technical Interview Corner: Tricky Questions & Drills

### Q1: The `as_index=False` Flag
**Question**: Why should you use `df.groupby('Department', as_index=False)['Salary'].mean()` instead of `df.groupby('Department')['Salary'].mean().reset_index()`?

**Answer**:
- `as_index=False` instructs the SQL-like grouping engine to treat group keys as ordinary columns rather than setting them as the DataFrame index.
- It is more concise, avoids allocating an intermediate index structure, and directly produces a flat tabular DataFrame.

In [11]:
# Demonstration of as_index=False
display(fortune.groupby("Sector", as_index=False)["Revenues"].sum().head(3))

,Sector,Revenues
0,Aerospace & Defense,383835.0
1,Apparel,101157.3
2,Business Services,316090.0


### Q2: Z-Score Standardization by Group
**Question**: An interviewer asks: *"How do you calculate the z-score of a metric within each group using vectorized Pandas?"*
$$\text{Z-Score} = \frac{x - \mu_{\text{group}}}{\sigma_{\text{group}}}$$

**Answer**:
Combine `.groupby()` and `.transform()`:
```python
df['z_score'] = (df['val'] - df.groupby('group')['val'].transform('mean')) / df.groupby('group')['val'].transform('std')
```

In [12]:
# Z-score of company revenues within their respective sector
rev_mean = fortune.groupby("Sector")["Revenues"].transform("mean")
rev_std = fortune.groupby("Sector")["Revenues"].transform("std")

fortune["Revenue_ZScore"] = ((fortune["Revenues"] - rev_mean) / rev_std).round(2)
fortune[["Company", "Sector", "Revenues", "Revenue_ZScore"]].head(5)

,Company,Sector,Revenues,Revenue_ZScore
0,Walmart,Retailing,500343.0,7.72
1,Exxon Mobil,Energy,244363.0,7.80
2,Berkshire Hathaway,Financials,242137.0,8.02
3,Apple,Technology,229234.0,7.48
4,UnitedHealth Group,Health Care,201159.0,4.97


### Q3: Advanced Interview Challenge: Market Dominance Ratio
**Challenge**: In a single chained expression, find the top 3 most monopolistic sectors—defined as sectors where the **#1 largest company controls the highest percentage of the total sector revenue**!

In [13]:
# Solution to Coding Challenge
monopoly_sectors = (
    fortune_enhanced.sort_values(by="Market_Share_Pct", ascending=False)
    .drop_duplicates(subset=["Sector"])
    [["Sector", "Company", "Revenues", "Sector_Total_Rev", "Market_Share_Pct"]]
    .sort_values(by="Market_Share_Pct", ascending=False)
    .head(3)
    .reset_index(drop=True)
)

print("Top 3 Most Concentrated Sectors in Fortune 1000:")
display(monopoly_sectors)

Top 3 Most Concentrated Sectors in Fortune 1000:


,Sector,Company,Revenues,Sector_Total_Rev,Market_Share_Pct
0,Motor Vehicles & Parts,General Motors,157311.0,433535.0,36.29
1,Telecommunications,AT&T,160546.0,466959.0,34.38
2,Apparel,Nike,34350.0,101157.3,33.96
